# 03 — Feature engineering

In [15]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

from fraud.data import load_train
from fraud.split import time_split

In [16]:
df = load_train()
train, valid, _ = time_split(df)

In [17]:
NON_FEATURES = ["isFraud", "TransactionDT", "TransactionID"]

MODEL_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.5,
    n_jobs=8,
    verbose=-1,
)

def evaluate(train, valid, seed=42):
    """Fit on train, score on valid."""
    model = lgb.LGBMClassifier(**MODEL_PARAMS,random_state=seed)
    model.fit(train.drop(columns=NON_FEATURES), train['isFraud'])
    scores = model.predict_proba(valid.drop(columns=NON_FEATURES))[:, 1]
    roc = roc_auc_score(valid["isFraud"], scores)
    pr = average_precision_score(valid["isFraud"], scores)
    print(f"seed {seed:<3} ROC-AUC {roc:.4f}   PR-AUC {pr:.4f}")

evaluate(train, valid)

seed 42  ROC-AUC 0.9247   PR-AUC 0.5884


In [18]:
for seed in [1, 2]:
    evaluate(train, valid, seed)

seed 1   ROC-AUC 0.9213   PR-AUC 0.5836
seed 2   ROC-AUC 0.9254   PR-AUC 0.5941


## 1. Time features

In [19]:
from fraud.features import add_time_features

df = add_time_features(df)
train, valid, _ = time_split(df)

In [20]:
(train.groupby("hour")["isFraud"].mean() * 100).round(1)

hour
0      3.2
1      3.3
2      3.6
3      3.5
4      5.0
5      6.2
6      7.6
7      9.7
8     10.0
9      9.6
10     7.5
11     4.1
12     3.1
13     2.2
14     2.4
15     2.5
16     3.0
17     3.2
18     3.5
19     3.5
20     3.4
21     3.4
22     3.3
23     3.9
Name: isFraud, dtype: float64

In [21]:
(train.groupby("weekday")["isFraud"].mean() * 100).round(1)

weekday
0    3.8
1    3.6
2    3.7
3    3.7
4    3.2
5    3.3
6    3.4
Name: isFraud, dtype: float64

In [22]:
for seed in [42, 1, 2]:
    evaluate(train, valid, seed)

seed 42  ROC-AUC 0.9222   PR-AUC 0.5835
seed 1   ROC-AUC 0.9222   PR-AUC 0.5907
seed 2   ROC-AUC 0.9220   PR-AUC 0.5891


`hour` and `weekday` are computed from a single row, so nothing is learned from other rows and they cannot leak.
The fraud rate differs strongly by hour on train (2.2% to 10.0%), but adding both features does not improve the
model: mean PR-AUC over three seeds is 0.589 without them and 0.588 with them, well within the seed noise of about
0.01. They stay out of the main model and will be re-checked for logistic regression and the lite demo model.

## 2. Simple row features

In [23]:
from fraud.features import add_row_features

df = df.drop(columns=["hour", "weekday"])
df = add_row_features(df)
train, valid, _ = time_split(df)

In [24]:
(train.groupby("amount_3dec")["isFraud"].mean() * 100).round(2)

amount_3dec
False     2.57
True     11.24
Name: isFraud, dtype: float64

In [25]:
(train.groupby("ProductCD", observed=True)["amount_3dec"].mean() * 100).round(1)

ProductCD
C    90.4
H     0.0
R     0.0
S     0.0
W     0.0
Name: amount_3dec, dtype: float64

In [26]:
(train.groupby("has_identity")["isFraud"].mean() * 100).round(2)

has_identity
False    2.14
True     7.34
Name: isFraud, dtype: float64

In [27]:
CANDIDATES = ["amount_cents", "amount_3dec", "has_identity", "n_missing"]

for name in CANDIDATES:
    others = [c for c in CANDIDATES if c != name]
    print(name)
    for seed in [42, 1, 2]:
        evaluate(train.drop(columns=others), valid.drop(columns=others), seed)

amount_cents
seed 42  ROC-AUC 0.9233   PR-AUC 0.5898
seed 1   ROC-AUC 0.9205   PR-AUC 0.5892
seed 2   ROC-AUC 0.9230   PR-AUC 0.5840
amount_3dec
seed 42  ROC-AUC 0.9221   PR-AUC 0.5867
seed 1   ROC-AUC 0.9211   PR-AUC 0.5871
seed 2   ROC-AUC 0.9234   PR-AUC 0.5873
has_identity
seed 42  ROC-AUC 0.9199   PR-AUC 0.5884
seed 1   ROC-AUC 0.9206   PR-AUC 0.5905
seed 2   ROC-AUC 0.9224   PR-AUC 0.5865
n_missing
seed 42  ROC-AUC 0.9190   PR-AUC 0.5828
seed 1   ROC-AUC 0.9209   PR-AUC 0.5879
seed 2   ROC-AUC 0.9233   PR-AUC 0.5915


Four simple row features were tested one at a time: cents of the amount, a third decimal digit, an identity flag
and the number of missing fields. Some look strong on their own (a third decimal: 11.2% fraud vs 2.6%; identity
present: 7.3% vs 2.1%), but none improves the model: mean PR-AUC over three seeds is 0.587–0.588 against the
baseline 0.589, within the seed noise. The model already sees this information: a third decimal occurs only in
product `C` (90% of its amounts), and identity presence and missingness are visible as NaN in the original
columns. All four are dropped for LightGBM; they may still help logistic regression.

In [28]:
df = df.drop(columns=CANDIDATES)

## 3. Card start day

In [29]:
print(f"D1 mean: train {train['D1'].mean():.0f}, valid {valid['D1'].mean():.0f}")

D1 mean: train 88, valid 111


In [30]:
from fraud.features import add_card_start_day

df = add_card_start_day(df)
train, valid, _ = time_split(df)

for seed in [42, 1, 2]:
    evaluate(train, valid, seed)

seed 42  ROC-AUC 0.9235   PR-AUC 0.6029
seed 1   ROC-AUC 0.9246   PR-AUC 0.6017
seed 2   ROC-AUC 0.9235   PR-AUC 0.6014


`card_start_day` is the transaction day minus `D1`. `D1` grows with time (mean 88 on train vs 111 on valid);
Kaggle participants read it as days since the card was first used, so the difference stays the same for all
transactions of one card and helps the model recognise the card. This matters because labels are propagated:
after a chargeback, later transactions linked to the same account are also marked as fraud. Mean PR-AUC over
three seeds rises from 0.589 to 0.602 and every seed beats the best baseline seed, while ROC-AUC stays at 0.924.
The feature is kept.